# **Hybrid RAG Search System**

**Problem: Develop a RAG system that combines keyword-based retrieval and semantic retrieval.**                                                                          
The system should compare:                                                   

*  Keyword retrieval
*  Vector/semantic retrieval
*   Hybrid retrieval

Students must demonstrate which approach performs better for different types of queries.                                                                        

**Viva Focus:**                                                            
BM25/keyword search                                                             
Embeddings                                                                      
Cosine similarity                                                                
Hybrid retrieval                                                                    
Precision and recall                                               


**Overall flow**

                    ┌───────────────────────┐
                    │       DOCUMENTS       │
                    │    Knowledge Base     │
                    └───────────┬───────────┘
                                │
                                │
                    ┌───────────▼───────────┐
                    │       USER QUERY       │
                    └───────────┬───────────┘
                                │
                 ┌──────────────┴──────────────┐
                 │                             │
                 ▼                             ▼
      ┌───────────────────┐         ┌─────────────────────┐
      │  KEYWORD SEARCH   │         │   SEMANTIC SEARCH   │
      │     (TF-IDF)      │         │    (EMBEDDINGS)     │
      └─────────┬─────────┘         └──────────┬──────────┘
                │                              │
                └──────────────┬───────────────┘
                               ▼
                  ┌─────────────────────────┐
                  │   HYBRID RETRIEVAL      │
                  │  Keyword + Semantic     │
                  └────────────┬────────────┘
                               │
                               ▼
                  ┌─────────────────────────┐
                  │   TOP RELEVANT DOCS     │
                  └────────────┬────────────┘
                               │
                               ▼
                  ┌─────────────────────────┐
                  │      RAG CONTEXT        │
                  └────────────┬────────────┘
                               │
                               ▼
                  ┌─────────────────────────┐
                  │         ANSWER          │
                  └─────────────────────────┘

**Install Required Libraries**

In [1]:
!pip install -q sentence-transformers scikit-learn pandas numpy

**Import Libraries**

In [2]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

**Create a Small Document Dataset**

In [3]:
documents = [
    "Machine learning allows computers to learn patterns from data.",

    "Deep learning uses neural networks to solve complex problems.",

    "Python is widely used for data science, machine learning and artificial intelligence.",

    "SQL is used to store, retrieve and manage data in relational databases.",

    "Natural language processing helps computers understand human language.",

    "Computer vision enables computers to understand images and videos.",

    "Retrieval Augmented Generation combines information retrieval with language generation.",

    "Embeddings represent text as numerical vectors that capture semantic meaning.",

    "Cosine similarity measures the similarity between two vectors.",

    "A vector database stores embeddings and enables semantic search.",

    "Data preprocessing includes cleaning missing values and removing duplicate records.",

    "Classification algorithms predict categories such as spam or not spam.",

    "Regression algorithms predict continuous numerical values.",

    "Random forests combine multiple decision trees to improve prediction performance.",

    "Large language models can generate human-like text from prompts."
]

print("Number of documents:", len(documents))

Number of documents: 15


**Display the Documents**

In [4]:
df = pd.DataFrame({
    "Document_ID": range(1, len(documents) + 1),
    "Document": documents
})

df

,Document_ID,Document
0,1,Machine learning allows computers to learn pat...
1,2,Deep learning uses neural networks to solve co...
2,3,"Python is widely used for data science, machin..."
3,4,"SQL is used to store, retrieve and manage data..."
4,5,Natural language processing helps computers un...
5,6,Computer vision enables computers to understan...
6,7,Retrieval Augmented Generation combines inform...
7,8,Embeddings represent text as numerical vectors...
8,9,Cosine similarity measures the similarity betw...
9,10,A vector database stores embeddings and enable...


# **PART 1 — KEYWORD RETRIEVAL**

Keyword retrieval searches for documents containing words related to the query.

**Create TF-IDF Matrix**

In [5]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (15, 102)


The rows represent documents and the columns represent words.

**Keyword Retrieval Function**

In [6]:
def keyword_search(query, top_k=5):

    query_vector = tfidf_vectorizer.transform([query])

    scores = cosine_similarity(query_vector, tfidf_matrix)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "Document_ID": index + 1,
            "Document": documents[index],
            "Keyword_Score": scores[index]
        })

    return pd.DataFrame(results)

**Test Keyword Retrieval**

In [27]:
query = "How does machine learning learn from data?"

keyword_results = keyword_search(query)

keyword_results

,Document_ID,Document,Keyword_Score
0,1,Machine learning allows computers to learn pat...,0.737843
1,3,"Python is widely used for data science, machin...",0.311513
2,15,Large language models can generate human-like ...,0.133452
3,2,Deep learning uses neural networks to solve co...,0.111765
4,11,Data preprocessing includes cleaning missing v...,0.088343


# **PART 2 — SEMANTIC / VECTOR RETRIEVAL**

Keyword search depends heavily on matching words.

Semantic search understands meaning using embeddings.

**Load Embedding Model**

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

This model converts text into numerical vectors.

**Generate Document Embeddings**

In [9]:
document_embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

print("Embedding shape:", document_embeddings.shape)

Embedding shape: (15, 384)


Each document is represented by a vector.

**Semantic Search Function**

In [10]:
def semantic_search(query, top_k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "Document_ID": index + 1,
            "Document": documents[index],
            "Semantic_Score": scores[index]
        })

    return pd.DataFrame(results)

**Test Semantic Search**

In [11]:
semantic_results = semantic_search(query)

semantic_results

,Document_ID,Document,Semantic_Score
0,1,Machine learning allows computers to learn pat...,0.767540
1,2,Deep learning uses neural networks to solve co...,0.392354
2,6,Computer vision enables computers to understan...,0.374741
3,12,Classification algorithms predict categories s...,0.350768
4,5,Natural language processing helps computers un...,0.343619


The semantic search can find documents based on meaning, even when the exact query words are not present.

# **PART 3 — HYBRID RETRIEVAL**

**Keyword Score
       +
Semantic Score
       =
Hybrid Score**

Hybrid Score =
0.5 × Keyword Score +
0.5 × Semantic Score

This gives equal importance to both methods.

**Hybrid Search Function**

In [12]:
def hybrid_search(query, top_k=5, alpha=0.5):

    # Keyword score
    query_tfidf = tfidf_vectorizer.transform([query])

    keyword_scores = cosine_similarity(
        query_tfidf,
        tfidf_matrix
    )[0]

    # Semantic score
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Hybrid score
    hybrid_scores = (
        alpha * keyword_scores +
        (1 - alpha) * semantic_scores
    )

    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "Document_ID": index + 1,
            "Document": documents[index],
            "Keyword_Score": round(keyword_scores[index], 4),
            "Semantic_Score": round(semantic_scores[index], 4),
            "Hybrid_Score": round(hybrid_scores[index], 4)
        })

    return pd.DataFrame(results)

**Test Hybrid Search**

In [13]:
hybrid_results = hybrid_search(query)

hybrid_results

,Document_ID,Document,Keyword_Score,Semantic_Score,Hybrid_Score
0,1,Machine learning allows computers to learn pat...,0.7378,0.7675,0.7527
1,3,"Python is widely used for data science, machin...",0.3115,0.2837,0.2976
2,2,Deep learning uses neural networks to solve co...,0.1118,0.3924,0.2521
3,6,Computer vision enables computers to understan...,0.0000,0.3747,0.1874
4,15,Large language models can generate human-like ...,0.1335,0.2216,0.1775


# **PART 4 — COMPARE ALL THREE METHODS**

**Create Test Queries**

In [14]:
queries = [
    "machine learning from data",
    "how computers understand human language",
    "storing data in databases",
    "understanding pictures using computers",
    "text generation using AI"
]

for q in queries:
    print("\nQUERY:", q)

    print("\nKeyword Search:")
    print(keyword_search(q, 3)[["Document_ID", "Document", "Keyword_Score"]])

    print("\nSemantic Search:")
    print(semantic_search(q, 3)[["Document_ID", "Document", "Semantic_Score"]])

    print("\nHybrid Search:")
    print(hybrid_search(q, 3)[["Document_ID", "Document", "Hybrid_Score"]])


QUERY: machine learning from data

Keyword Search:
   Document_ID                                           Document  \
0            1  Machine learning allows computers to learn pat...   
1            3  Python is widely used for data science, machin...   
2           15  Large language models can generate human-like ...   

   Keyword_Score  
0       0.627110  
1       0.366519  
2       0.157017  

Semantic Search:
   Document_ID                                           Document  \
0            1  Machine learning allows computers to learn pat...   
1           12  Classification algorithms predict categories s...   
2           13  Regression algorithms predict continuous numer...   

   Semantic_Score  
0        0.691801  
1        0.364026  
2        0.347948  

Hybrid Search:
   Document_ID                                           Document  \
0            1  Machine learning allows computers to learn pat...   
1            3  Python is widely used for data science, machin... 

# **PART 5 — PRECISION AND RECALL**

For evaluation, we need to define which documents are actually relevant.

**Define Relevant Documents**

We can define the relevant document IDs manually.

In [15]:
relevant_docs = {
    "machine learning from data": {1, 3, 14},

    "how computers understand human language": {5},

    "storing data in databases": {4, 10},

    "understanding pictures using computers": {6},

    "text generation using AI": {7, 15}
}

These are our ground-truth relevant documents for this small demonstration dataset.

**Precision and Recall Function**

In [16]:
def calculate_precision_recall(retrieved_docs, relevant_docs):

    retrieved_set = set(retrieved_docs)
    relevant_set = set(relevant_docs)

    true_positive = len(retrieved_set & relevant_set)

    precision = (
        true_positive / len(retrieved_set)
        if len(retrieved_set) > 0 else 0
    )

    recall = (
        true_positive / len(relevant_set)
        if len(relevant_set) > 0 else 0
    )

    return precision, recall

**Formulas**

### Precision

$$
Precision = \frac{Relevant\ Retrieved\ Documents}{Total\ Retrieved\ Documents}
$$

### Recall

$$
Recall = \frac{Relevant\ Retrieved\ Documents}{Total\ Relevant\ Documents}
$$

**Evaluate Keyword Search**

In [17]:
query = "machine learning from data"

results = keyword_search(query, top_k=3)

retrieved_ids = results["Document_ID"].tolist()

precision, recall = calculate_precision_recall(
    retrieved_ids,
    relevant_docs[query]
)

print("Keyword Retrieval")
print("Retrieved:", retrieved_ids)
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))

Keyword Retrieval
Retrieved: [1, 3, 15]
Precision: 0.67
Recall: 0.67


**Evaluate Semantic Search**

In [18]:
query = "machine learning from data"

results = semantic_search(query, top_k=3)

retrieved_ids = results["Document_ID"].tolist()

precision, recall = calculate_precision_recall(
    retrieved_ids,
    relevant_docs[query]
)

print("Semantic Retrieval")
print("Retrieved:", retrieved_ids)
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))

Semantic Retrieval
Retrieved: [1, 12, 13]
Precision: 0.33
Recall: 0.33


**Evaluate Hybrid Search**

In [19]:
query = "machine learning from data"

results = hybrid_search(query, top_k=3)

retrieved_ids = results["Document_ID"].tolist()

precision, recall = calculate_precision_recall(
    retrieved_ids,
    relevant_docs[query]
)

print("Hybrid Retrieval")
print("Retrieved:", retrieved_ids)
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))

Hybrid Retrieval
Retrieved: [1, 3, 2]
Precision: 0.67
Recall: 0.67


# **PART 6 — Compare Precision and Recall Automatically**

Instead of checking only one query, let's evaluate all queries.

**Evaluation Function**

In [20]:
def evaluate_all_methods(queries, top_k=3):

    results = []

    for query in queries:

        relevant = relevant_docs[query]

        # Keyword
        keyword_result = keyword_search(query, top_k)
        keyword_ids = keyword_result["Document_ID"].tolist()

        kp, kr = calculate_precision_recall(
            keyword_ids,
            relevant
        )

        # Semantic
        semantic_result = semantic_search(query, top_k)
        semantic_ids = semantic_result["Document_ID"].tolist()

        sp, sr = calculate_precision_recall(
            semantic_ids,
            relevant
        )

        # Hybrid
        hybrid_result = hybrid_search(query, top_k)
        hybrid_ids = hybrid_result["Document_ID"].tolist()

        hp, hr = calculate_precision_recall(
            hybrid_ids,
            relevant
        )

        results.append({
            "Query": query,
            "Keyword Precision": kp,
            "Keyword Recall": kr,
            "Semantic Precision": sp,
            "Semantic Recall": sr,
            "Hybrid Precision": hp,
            "Hybrid Recall": hr
        })

    return pd.DataFrame(results)

**Run Evaluation**

In [21]:
evaluation_results = evaluate_all_methods(queries)

evaluation_results

,Query,Keyword Precision,Keyword Recall,Semantic Precision,Semantic Recall,Hybrid Precision,Hybrid Recall
0,machine learning from data,0.666667,0.666667,0.333333,0.333333,0.666667,0.666667
1,how computers understand human language,0.333333,1.000000,0.333333,1.000000,0.333333,1.000000
2,storing data in databases,0.333333,0.500000,0.666667,1.000000,0.666667,1.000000
3,understanding pictures using computers,0.333333,1.000000,0.333333,1.000000,0.333333,1.000000
4,text generation using AI,0.666667,1.000000,0.666667,1.000000,0.666667,1.000000


# **PART 7 — SIMPLE RAG**

Now we have the retrieval part.

A RAG system has two major stages:

             ┌───────────────────┐
             │     RETRIEVAL     │
             └─────────┬─────────┘
                       ↓
             ┌───────────────────┐
             │ RELEVANT DOCUMENTS│
             └─────────┬─────────┘
                       ↓
             ┌───────────────────┐
             │    GENERATION     │
             └─────────┬─────────┘
                       ↓
             ┌───────────────────┐
             │      ANSWER       │
             └───────────────────┘

**Create RAG Context**

In [22]:
def create_context(query, top_k=3):

    results = hybrid_search(query, top_k)

    context = "\n".join(
        results["Document"].tolist()
    )

    return context

**Test RAG Retrieval**

In [23]:
query = "What is machine learning?"

context = create_context(query)

print("Retrieved Context:\n")
print(context)

Retrieved Context:

Machine learning allows computers to learn patterns from data.
Python is widely used for data science, machine learning and artificial intelligence.
Deep learning uses neural networks to solve complex problems.


The retrieved documents become the context supplied to a generator/LLM.

**Simple RAG Answer Function**

For a completely local/simple demonstration, we can return the retrieved context as the evidence.

In [24]:
def simple_rag(query, top_k=3):

    results = hybrid_search(query, top_k)

    print("QUERY:")
    print(query)

    print("\nRETRIEVED DOCUMENTS:")

    for i, row in results.iterrows():
        print(f"\n{i+1}. {row['Document']}")

    print("\nRAG CONTEXT:")
    print("\n".join(results["Document"].tolist()))

**Run RAG**

In [25]:
simple_rag(
    "What is machine learning?",
    top_k=3
)

QUERY:
What is machine learning?

RETRIEVED DOCUMENTS:

1. Machine learning allows computers to learn patterns from data.

2. Python is widely used for data science, machine learning and artificial intelligence.

3. Deep learning uses neural networks to solve complex problems.

RAG CONTEXT:
Machine learning allows computers to learn patterns from data.
Python is widely used for data science, machine learning and artificial intelligence.
Deep learning uses neural networks to solve complex problems.


This demonstrates the Retrieval part of RAG without requiring an API key.

# **PART 8 — Try Different Query Types**

**Query Comparison**

In [26]:
test_queries = [
    "machine learning",
    "computers understanding human language",
    "database data storage",
    "AI understanding images",
    "generating human-like text"
]

for query in test_queries:

    print("=" * 80)
    print("QUERY:", query)

    print("\nKEYWORD:")
    print(keyword_search(query, 2)[["Document_ID", "Document"]])

    print("\nSEMANTIC:")
    print(semantic_search(query, 2)[["Document_ID", "Document"]])

    print("\nHYBRID:")
    print(hybrid_search(query, 2)[["Document_ID", "Document"]])

QUERY: machine learning

KEYWORD:
   Document_ID                                           Document
0            1  Machine learning allows computers to learn pat...
1            3  Python is widely used for data science, machin...

SEMANTIC:
   Document_ID                                           Document
0            1  Machine learning allows computers to learn pat...
1           12  Classification algorithms predict categories s...

HYBRID:
   Document_ID                                           Document
0            1  Machine learning allows computers to learn pat...
1            3  Python is widely used for data science, machin...
QUERY: computers understanding human language

KEYWORD:
   Document_ID                                           Document
0            5  Natural language processing helps computers un...
1           15  Large language models can generate human-like ...

SEMANTIC:
   Document_ID                                           Document
0            5  Natur